# Heart Disease Prediction - Logistic Regression (Day 3)

This notebook covers the training, evaluation, and coefficient analysis of a Logistic Regression model trained on the preprocessed combined UCI Heart Disease dataset.

## Step 1: Import Libraries

**What & Why:**
- **What:** Imported data handling, modeling, evaluation metrics, and plotting libraries.
- **Why:** These provide the necessary modules for model instantiation, fitting, performance metrics computation (accuracy, recall, precision, confusion matrix), and plotting coefficient weights.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

## Step 2: Load the Dataset

**What & Why:**
- **What:** Loaded `heart_processed.csv` (imputed and encoded dataset) from the `data/` directory.
- **Why:** To read the fully cleaned tabular records before partitioning into features and labels.

In [ ]:
df = pd.read_csv("../data/heart_processed.csv")
df.head()

## Step 3: Split Features and Target

**What & Why:**
- **What:** Separated attributes (`X`) from class labels (`y`).
- **Why:** Required formatting to pass arguments to scikit-learn's split and fit functions.

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

## Step 4: Train-Test Split

**What & Why:**
- **What:** Executed an 80/20 train-test partition using stratification.
- **Why:** To isolate the test set from training to evaluate model generalization fairly.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Step 5: Feature Scaling

**What & Why:**
- **What:** Normalised feature values using standard scaling.
- **Why:** Logistic regression calculates feature weights/coefficients and uses gradient solvers. Scaling prevents wide-scale features from dominating parameter updates.

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Step 6: Create the Logistic Regression Model

**What & Why:**
- **What:** Instantiated the `LogisticRegression` estimator.
- **Why:** We fix `random_state=42` to guarantee reproducible solver coefficients across runs.

In [ ]:
model = LogisticRegression(random_state=42)

## Step 7: Train the Model

**What & Why:**
- **What:** Fitted the logistic regression model on the scaled training dataset.
- **Why:** To optimize weights and bias values to estimate log-odds probabilities of disease presence.

In [ ]:
model.fit(X_train, y_train)

## Step 8: Make Predictions

**What & Why:**
- **What:** Generated target predictions on test features (`X_test`) and displayed actual vs. predicted values.
- **Why:** To perform quality checks on predictions before computing general metric scores.

In [ ]:
y_pred = model.predict(X_test)

# Check a few predictions
predictions = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
predictions.head(10)

## Step 9: Calculate Accuracy

**What, Why & Observations:**
- **What:** Computed test accuracy score.
- **Why:** Gives a baseline score of overall correctness across all patients.
- **Observations:** The model achieved **82.07% accuracy**, indicating high overall performance.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

## Step 10: Confusion Matrix

**What, Why & Observations:**
- **What:** Printed and plotted the confusion matrix showing true positives, false positives, false negatives, and true negatives.
- **Why:** Accuracy alone can mask errors. The matrix shows exactly where prediction mistakes occurred.
- **Observations:** 
  - 65 patients were correctly predicted as healthy (True Negatives).
  - 86 patients were correctly predicted as diseased (True Positives).
  - 17 patients were incorrectly predicted as diseased (False Positives).
  - 16 patients were incorrectly predicted as healthy (False Negatives).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Visualize the Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

## Step 11: Classification Report

**What, Why & Observations:**
- **What:** Generated a classification report displaying precision, recall, and F1-score per target class.
- **Why:** In medical diagnostic cases, precision and recall per target class are critical metrics.
- **Observations:** Class 1 (Disease) shows slightly higher recall (0.84) and precision (0.83) compared to class 0 (0.79 and 0.80 respectively).

In [ ]:
print(classification_report(y_test, y_pred))

## Step 12: Learn the Metrics

**What & Why:**
- **What:** Documented definitions for Accuracy, Precision, Recall, and F1-Score.
- **Why:** To make the notebook professional and establish clear clinical contexts for metric scores.

### Metric Definitions:

#### 1. Accuracy
- **Formula:** `(TP + TN) / (TP + TN + FP + FN)`
- **Interpretation:** The proportion of total patients predicted correctly.

#### 2. Precision
- **Formula:** `TP / (TP + FP)`
- **Interpretation:** When the model predicts Heart Disease, how often is it correct? High precision minimizes false alarms.

#### 3. Recall (Sensitivity)
- **Formula:** `TP / (TP + FN)`
- **Interpretation:** Out of all actual heart disease patients, how many did we detect? Critical in healthcare to avoid missing sick patients.

#### 4. F1-Score
- **Formula:** `2 * (Precision * Recall) / (Precision + Recall)`
- **Interpretation:** Harmonic mean balancing precision and recall.

## Step 13: Model Coefficients

**What, Why & Observations:**
- **What:** Extracted and plotted feature coefficients.
- **Why:** Logistic regression coefficients show the directional impact and magnitude of features on disease likelihood, offering high model interpretability.
- **Observations:**
  - `ca` (colored vessels) and `exang` (exercise angina) show the strongest positive coefficients, meaning they correlate with higher risk.
  - `cp` (chest pain) and `chol` (cholesterol) show the strongest negative coefficients in the label-encoded feature set.

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

# Sort by Importance
importance = importance.sort_values(
    by="Coefficient",
    ascending=False
)
print(importance)

# Visualize Feature Importance
plt.figure(figsize=(10,6))
plt.barh(
    importance["Feature"],
    importance["Coefficient"]
)
plt.title("Logistic Regression Feature Importance")
plt.xlabel("Coefficient")
plt.show()

## 📝 Mini Exercise Questions & Answers

**1. What accuracy did the model achieve?**
- The model achieved an accuracy of **82.07%** (0.82).

**2. How many patients were predicted correctly?**
- **151 patients** were predicted correctly out of the 184 test patients (65 True Negatives + 86 True Positives).

**3. Which metric is most important in medical diagnosis: Accuracy, Precision, or Recall? Why?**
- **Recall** is the most critical metric in medical diagnostics. Recall measures the model's ability to identify all actual positive cases (heart disease patients). A False Negative (missing a patient with heart disease) means leaving a sick patient untreated, which has severe, potentially fatal consequences. A False Positive (calling a healthy patient diseased) will easily be caught in follow-up screening.

**4. Which feature had the largest positive coefficient?**
- **`ca`** (number of major vessels colored by fluoroscopy) with a coefficient value of **0.58**.

**5. Which feature had the largest negative coefficient?**
- **`cp`** (chest pain type) with a coefficient value of **-0.50**.

**6. Did the model perform equally well for both classes?**
- Yes, performance was balanced. For the healthy class (`0`), the model achieved an F1-score of **0.80** (Precision: 0.80, Recall: 0.79), and for the diseased class (`1`), it achieved an F1-score of **0.84** (Precision: 0.83, Recall: 0.84).